In [ ]:
!git clone --branch evaluation-pipeline https://github.com/tarun1125/CSAIML-Capstone-Project-20.git


In [5]:
%pip install transformers accelerate torch bitsandbytes sentencepiece pymongo huggingface_hub codebleu bert-score tree-sitter tree-sitter-python

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/38.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.0 MB ? eta -:--:--
    --------------------------------------- 0.5/38.0 MB 590.3 kB/s eta 0:01:04
    --------------------------------------- 0.8/38.0 MB 618.5 kB/s eta 0:01:01
    --------------------------------------- 0.8/38.0 MB 618.5 kB/s eta 0:01:01
    --------------------------------------- 0.8/38.0 MB 618.5 kB/s eta 0:01:01
   - -------------------------------------- 1.0/38.0 MB 594.2 kB/s eta 0:01:03
   - -------------------------------------- 1.0/38.0 MB 594.2 kB/s eta 0:01:03
   - -------------------------------------- 1.3/38.0 MB 599.3 kB/s eta 0:01:02
   - --------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [9]:
cd /content/CSAIML-Capstone-Project-20

/content/CSAIML-Capstone-Project-20


In [10]:
import os

from google.colab import userdata
HF_TOKEN   = userdata.get("HF_TOKEN")
ATLAS_URI  = userdata.get("MONGODB_URI")

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("capstone-eval")


In [ ]:
import json
import time
from pathlib import Path

SYSTEM_PROMPT = """You are a MongoDB query expert.
When given a natural-language question and a database schema, you output ONLY the raw PyMongo query — no explanation, no markdown, no prose.

Schema (6 databases, 27 collections -- the full-schema arm; compare later
against the retrieved-schema RAG arm):

concert_singer:
- stadium: { Stadium_ID (int), Location (str), Name (str), Capacity (int), Highest (int), Lowest (int), Average (int) }
- concert: { concert_ID (int), concert_Name (str), Theme (str), Stadium_ID (str), Year (str) }
- singer: { Singer_ID (int), Name (str), Country (str), Song_Name (str), Song_release_year (str), Age (int), Is_male (str) }
- singer_in_concert: { concert_ID (int), Singer_ID (str) }

pets_1:
- Student: { StuID (int), LName (str), Fname (str), Age (int), Sex (str), Major (int), Advisor (int), city_code (str) }
- Has_Pet: { StuID (int), PetID (int) }
- Pets: { PetID (int), PetType (str), pet_age (int), weight (float) }

network_1:
- Friend: { student_id (int), friend_id (int) }
- Highschooler: { ID (int), name (str), grade (int) }
- Likes: { student_id (int), liked_id (int) }

car_1:
- continents: { ContId (int), Continent (str) }
- car_makers: { Id (int), Maker (str), FullName (str), Country (str) }
- countries: { CountryId (int), CountryName (str), Continent (int) }
- model_list.json: { ModelId (int), Maker (int), Model (str) }    (collection name literally contains ".json" -- access as db['model_list.json'], never db.model_list.json)
- cars_data: { Id (int), MPG (str), Cylinders (int), Edispl (float), Horsepower (str), Weight (int), Accelerate (float), Year (int) }
- car_names: { MakeId (int), Model (str), Make (str) }

world_1:
- city: { ID (int), Name (str), CountryCode (str), District (str), Population (int) }
- country: { Code (str), Name (str), Continent (str), Region (str), SurfaceArea (float), IndepYear (int, nullable), Population (int), LifeExpectancy (float, nullable), GNP (float), GNPOld (float, nullable), LocalName (str), GovernmentForm (str), HeadOfState (str, nullable), Capital (int, nullable), Code2 (str) }
- countrylanguage: { CountryCode (str), Language (str), IsOfficial (str), Percentage (float) }

dog_kennels:
- Breeds: { breed_code (str), breed_name (str) }
- Charges: { charge_id (int), charge_type (str), charge_amount (int) }
- Sizes: { size_code (str), size_description (str) }
- Treatment_Types: { treatment_type_code (str), treatment_type_description (str) }
- Owners: { owner_id (int), first_name (str), last_name (str), street (str), city (str), state (str), zip_code (str), email_address (str), home_phone (str), cell_number (str) }
- Dogs: { dog_id (int), owner_id (int), abandoned_yn (str), breed_code (str), size_code (str), name (str), age (str), date_of_birth (str), gender (str), weight (str), date_arrived (str), date_adopted (str), date_departed (str) }
- Professionals: { professional_id (int), role_code (str), first_name (str), street (str), city (str), state (str), zip_code (str), last_name (str), email_address (str), home_phone (str), cell_number (str) }
- Treatments: { treatment_id (int), dog_id (int), professional_id (int), treatment_type_code (str), date_of_treatment (str), cost_of_treatment (int) }

Note: several logically-numeric fields are stored as strings (e.g.
concert.Stadium_ID, Dogs.age, Dogs.weight, cars_data.Horsepower, cars_data.MPG,
singer_in_concert.Singer_ID) -- use $toInt / $toDouble when comparing or
aggregating on these.

Rules:
1. Output ONLY the PyMongo expression (e.g. list(db.singer.find({...})))
2. Use db.<collection>.<method>() syntax -- use db['model_list.json'] for that one collection
3. Do NOT wrap in ```python or any markdown
4. Do NOT add any explanation before or after"""

# -----------------------------------------------------------------------------
# Load all 121 test cases from the shared reference file instead of a
# hardcoded 12-question list -- reference_queries.json is the single source
# of truth this cell, execute_gold.py, and execute_queries.py all read from,
# so Qwen is tested on exactly the same cases Claude's arm and the gold
# answers were built against. Ids are carried through unchanged (a mix of
# int 1-12 and strings like "cs-e1") so no id-remapping table is needed
# downstream in cells 7/9.
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
if not (ROOT / "data" / "reference_queries.json").exists():
    ROOT = ROOT.parent
REFERENCE = ROOT / "data" / "reference_queries.json"

with open(REFERENCE, encoding="utf-8") as f:
    CASES = json.load(f)

log.info("Loaded %d test cases from %s", len(CASES), REFERENCE)

predictions_qwen = []

for i, case in enumerate(CASES, 1):
    qid = case["id"]
    nl = case["question"]
    database = case.get("database")

    log.info("[%d/%d] [%s] querying Qwen...", i, len(CASES), qid)
    t0 = time.time()

    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": nl}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.0,
            do_sample=False
        )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = raw.replace("```python", "").replace("```", "").strip()

        latency = time.time() - t0
        log.info("[%s] OK (%.2fs): %s", qid, latency, raw[:80])

        predictions_qwen.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": raw,
            "latency_s": round(latency, 3)
        })

    except Exception as e:
        log.error("[%s] FAILED: %s", qid, e)
        predictions_qwen.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": "",
            "error": str(e)
        })

# Save to data/qwen2.5-coder_results.json -- the exact path normalize.py
# defaults to, so `python normalize.py` with no args keeps working.
OUT_PATH = ROOT / "data" / "qwen2.5-coder_results.json"
OUT_PATH.parent.mkdir(exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions_qwen, f, indent=2)

log.info("Saved %d predictions -> %s", len(predictions_qwen), OUT_PATH)


In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "reference_queries.json").exists():
    ROOT = ROOT.parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Reads the honest fields evaluation/execute_queries.py already computes --
# execution_accuracy is compared against data/gold_results.json (order-
# insensitive), not just "did it run without throwing". status=="PASS" alone
# is ONLY that -- an empty or wrong-but-non-crashing result still says PASS,
# which was exactly bug B2 (see docs/BACKLOG.md #2).
RUNS = [
    ("Claude",  ROOT / "data" / "claude_execution_results.json"),
    ("Qwen2.5", ROOT / "data" / "qwen_execution_results.json"),
]

rows = []
for model_name, path in RUNS:
    if not path.exists():
        print(f"[{model_name}] {path} not found -- run evaluation/execute_queries.py locally first")
        continue

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    total = len(data)
    ran_ok = sum(x.get("status") == "PASS" for x in data)
    non_empty = sum(x.get("non_empty_rate") is True for x in data)
    correct = sum(x.get("execution_accuracy") is True for x in data)

    print(f"{model_name:10} ran_ok={ran_ok}/{total}  non_empty={non_empty}/{total}  "
          f"execution_accuracy={correct}/{total} ({100*correct/max(total,1):.1f}%)")

    rows.append({
        "Model": model_name,
        "Accuracy": round(100 * correct / max(total, 1), 2),
        "NonEmptyRate": round(100 * non_empty / max(total, 1), 2),
        "RanOK": round(100 * ran_ok / max(total, 1), 2),
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "execution_scores.csv", index=False)
print("\nSaved to:", OUTPUT_DIR / "execution_scores.csv")
print(df)


In [ ]:
# BLEU Score
from pathlib import Path
import json
import pandas as pd

from codebleu import calc_codebleu

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

ROOT = Path.cwd()
if not (ROOT / "data" / "reference_queries.json").exists():
    ROOT = ROOT.parent

REFERENCE = ROOT / "data/reference_queries.json"
CLAUDE = ROOT / "data/claude_normalized.json"
QWEN = ROOT / "data/qwen_normalized.json"

OUTPUT = ROOT / "outputs/codebleu_scores.csv"

# -----------------------------------------------------------------------------
# Load helpers -- no ID_MAP. reference_queries.json's own ids (mix of int
# 1-12 and str like "cs-e1") are what Qwen's cell 5 and Claude's arm both
# already carry through unchanged, so joining on str(id) is enough. The old
# ID_MAP only ever covered the original 12 "easy-1".."complex-3" ids and
# would KeyError on any of the 109 new ones.
# -----------------------------------------------------------------------------

def load_reference(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {str(x["id"]): x["normalized_query"] for x in data}


def load_model(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {str(row["id"]): row["normalized_query"] for row in data}


# -----------------------------------------------------------------------------
# Compute corpus CodeBLEU
# -----------------------------------------------------------------------------

def compute_codebleu(reference, prediction, model_name):
    shared_ids = sorted(set(reference) & set(prediction))
    missing = set(reference) - set(prediction)
    if missing:
        preview = sorted(missing)[:5]
        print(f"[{model_name}] WARNING: {len(missing)} case(s) have no prediction, skipped: {preview}{'...' if len(missing) > 5 else ''}")

    references = [[reference[qid]] for qid in shared_ids]
    predictions = [prediction[qid] for qid in shared_ids]

    result = calc_codebleu(
        references=references,
        predictions=predictions,
        lang="python",
    )

    # calc_codebleu's own composite computes theta * (dataflow_match_score or 1) --
    # when dataflow degenerates to 0.0 (no reference data-flow graphs for this
    # corpus), Python's `0.0 or 1` evaluates to 1, so the composite silently
    # substitutes a PERFECT dataflow score instead of penalizing the missing
    # metric. codebleu_3component is the honest number: mean of the 3
    # components that are actually meaningful here, dataflow excluded rather
    # than faked.
    three_component = (
        result["ngram_match_score"]
        + result["weighted_ngram_match_score"]
        + result["syntax_match_score"]
    ) / 3

    print("\n" + "=" * 60)
    print(model_name, f"  (n={len(shared_ids)})")
    print("=" * 60)
    for k, v in result.items():
        print(f"{k:30} {v:.4f}")
    print(f"{'codebleu_3component (honest)':30} {three_component:.4f}")

    return {
        "Model": model_name,
        "n": len(shared_ids),
        **{k: round(v, 4) for k, v in result.items()},
        "codebleu_3component": round(three_component, 4),
    }


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

reference = load_reference(REFERENCE)
claude = load_model(CLAUDE)
qwen = load_model(QWEN)

rows = []
rows.append(compute_codebleu(reference, claude, "Claude"))
rows.append(compute_codebleu(reference, qwen, "Qwen2.5"))

df = pd.DataFrame(rows)
OUTPUT.parent.mkdir(exist_ok=True)
df.to_csv(OUTPUT, index=False)

print("\nSaved to:", OUTPUT)
print(df[["Model", "n", "codebleu", "codebleu_3component"]])


In [3]:
%pip install bert-score torch transformers pandas

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# BERT SCORE
from pathlib import Path
import json

import pandas as pd
from bert_score import score


# --------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------

ROOT = Path.cwd()

if not (ROOT / "data" / "reference_queries.json").exists():
    ROOT = ROOT.parent

REFERENCE = ROOT / "data/reference_queries.json"
CLAUDE = ROOT / "data/claude_normalized.json"
QWEN = ROOT / "data/qwen_normalized.json"

OUTPUT = ROOT / "outputs/bertscore_scores.csv"

# --------------------------------------------------------------------
# Loaders -- no ID_MAP, same reasoning as cell 7: join on str(id) directly.
# --------------------------------------------------------------------

def load_reference():
    with open(REFERENCE, encoding="utf-8") as f:
        data = json.load(f)
    return {str(x["id"]): x["normalized_query"] for x in data}


def load_model(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return {str(row["id"]): row["normalized_query"] for row in data}


# --------------------------------------------------------------------
# Compute
# --------------------------------------------------------------------

def evaluate(model_name, predictions, reference):
    shared_ids = sorted(set(reference) & set(predictions))
    missing = set(reference) - set(predictions)
    if missing:
        print(f"[{model_name}] WARNING: {len(missing)} case(s) have no prediction, skipped")

    refs = [reference[qid] for qid in shared_ids]
    preds = [predictions[qid] for qid in shared_ids]

    print(f"\nEvaluating {model_name}... (n={len(shared_ids)})")

    P, R, F1 = score(
        preds,
        refs,
        lang="en",
        rescale_with_baseline=True,
    )

    rows = []
    for i, qid in enumerate(shared_ids):
        rows.append({
            "id": qid,
            "model": model_name,
            "precision": round(P[i].item(), 4),
            "recall": round(R[i].item(), 4),
            "f1": round(F1[i].item(), 4),
        })

    print(f"Average F1 ({model_name}) = {F1.mean().item():.4f}")
    return rows


# --------------------------------------------------------------------
# Main
# --------------------------------------------------------------------

reference = load_reference()
claude = load_model(CLAUDE)
qwen = load_model(QWEN)

rows = []
rows.extend(evaluate("Claude", claude, reference))
rows.extend(evaluate("Qwen2.5", qwen, reference))

df = pd.DataFrame(rows)
OUTPUT.parent.mkdir(exist_ok=True)
df.to_csv(OUTPUT, index=False)

print("\nSaved to:", OUTPUT)
print("\nAverage Scores")
print(df.groupby("model")[["precision", "recall", "f1"]].mean().round(4))


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUTPUT = ROOT / "outputs"
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(exist_ok=True)

# ---------------------------------------------------
# Load data -- all three read from files the earlier cells actually save,
# nothing typed in by hand. If a file's missing, the earlier cell hasn't
# been run yet -- fail loud instead of silently falling back to stale
# numbers.
# ---------------------------------------------------

for name in ("execution_scores.csv", "codebleu_scores.csv", "bertscore_scores.csv"):
    if not (OUTPUT / name).exists():
        raise FileNotFoundError(f"{OUTPUT / name} missing -- run cells 6, 7, and 9 first")

execution = pd.read_csv(OUTPUT / "execution_scores.csv")
codebleu = pd.read_csv(OUTPUT / "codebleu_scores.csv")
bertscore = pd.read_csv(OUTPUT / "bertscore_scores.csv")

# ---------------------------------------------------
# Figure 1: Execution Accuracy
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
plt.bar(execution["Model"], execution["Accuracy"])
plt.ylabel("Execution Accuracy (%)")
plt.title("Execution Accuracy Comparison")
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig(FIGURES / "execution_accuracy.png", dpi=300)
plt.close()

# ---------------------------------------------------
# Figure 2: CodeBLEU (honest 3-component, dataflow excluded)
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
plt.bar(codebleu["Model"], codebleu["codebleu_3component"])
plt.ylabel("CodeBLEU (3-component, dataflow excluded)")
plt.title("CodeBLEU Comparison")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES / "codebleu.png", dpi=300)
plt.close()

# ---------------------------------------------------
# Figure 3: BERTScore
# ---------------------------------------------------
avg = bertscore.groupby("model")[["precision", "recall", "f1"]].mean()
x = range(len(avg))
width = 0.25

plt.figure(figsize=(8, 5))
plt.bar([i - width for i in x], avg["precision"], width=width, label="Precision")
plt.bar(x, avg["recall"], width=width, label="Recall")
plt.bar([i + width for i in x], avg["f1"], width=width, label="F1")
plt.xticks(x, avg.index)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Average BERTScore")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "bertscore.png", dpi=300)
plt.close()

# ---------------------------------------------------
# Figure 4: Overall Comparison
# ---------------------------------------------------
overall = execution.merge(
    codebleu[["Model", "codebleu_3component"]], on="Model"
).merge(
    bertscore.groupby("model")["f1"].mean().rename("BERTScore_F1").reset_index().rename(columns={"model": "Model"}),
    on="Model"
)

x = range(len(overall))
width = 0.25

plt.figure(figsize=(9, 5))
plt.bar([i - width for i in x], overall["Accuracy"], width=width, label="Execution (%)")
plt.bar(x, overall["codebleu_3component"] * 100, width=width, label="CodeBLEU")
plt.bar([i + width for i in x], overall["BERTScore_F1"] * 100, width=width, label="BERTScore F1")
plt.xticks(x, overall["Model"])
plt.ylabel("Score")
plt.title("Overall Benchmark Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "overall_comparison.png", dpi=300)
plt.close()

print("Saved all figures to", FIGURES)
print(overall)
